# Data Collection

This notebook loads and explores the raw NBA player game log dataset used in the project.

At this stage, the goal is to:
- inspect the raw full-league dataset
- understand its structure
- verify data quality
- prepare a clean starting point for feature engineering

In [1]:
import pandas as pd

## Load raw dataset

The raw dataset contains historical NBA game logs for multiple players collected from the NBA API.

In [2]:
df = pd.read_csv("../data/raw/nba_players_2023_24_raw.csv")
df.head()

,Game_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG3M,FG3A,FTM,FTA,REB,AST,TOV,STL,BLK,PTS,PLUS_MINUS,PLAYER_NAME,PLAYER_ID
0,22300277,2023-12-01,DAL vs. MEM,L,4,0,1,0,1,1,2,1,0,0,0,0,1,4,A.J. Lawson,1630639
1,22300287,2023-12-02,DAL vs. OKC,L,19,4,10,3,7,1,2,0,2,0,0,1,12,9,A.J. Lawson,1630639
2,22301213,2023-12-06,DAL vs. UTA,W,7,2,2,0,0,0,0,1,0,0,0,0,4,7,A.J. Lawson,1630639
3,22301226,2023-12-08,DAL @ POR,W,7,1,3,0,2,0,0,1,0,0,0,0,2,-4,A.J. Lawson,1630639
4,22300299,2023-12-11,DAL @ MEM,W,14,2,7,0,4,0,0,1,1,1,0,0,4,1,A.J. Lawson,1630639


## Dataset overview

Check the size of the dataset, number of players, and basic date range.

In [3]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Players:", df["PLAYER_NAME"].nunique())

Rows: 15914
Columns: 20
Players: 277


## Column names

Review the available raw variables before feature engineering.

In [4]:
df.columns.tolist()

['Game_ID',
 'GAME_DATE',
 'MATCHUP',
 'WL',
 'MIN',
 'FGM',
 'FGA',
 'FG3M',
 'FG3A',
 'FTM',
 'FTA',
 'REB',
 'AST',
 'TOV',
 'STL',
 'BLK',
 'PTS',
 'PLUS_MINUS',
 'PLAYER_NAME',
 'PLAYER_ID']

## Data types and completeness

This helps verify whether columns have the expected types and whether there are missing values.

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15914 entries, 0 to 15913
Data columns (total 20 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Game_ID      15914 non-null  int64 
 1   GAME_DATE    15914 non-null  object
 2   MATCHUP      15914 non-null  object
 3   WL           15914 non-null  object
 4   MIN          15914 non-null  int64 
 5   FGM          15914 non-null  int64 
 6   FGA          15914 non-null  int64 
 7   FG3M         15914 non-null  int64 
 8   FG3A         15914 non-null  int64 
 9   FTM          15914 non-null  int64 
 10  FTA          15914 non-null  int64 
 11  REB          15914 non-null  int64 
 12  AST          15914 non-null  int64 
 13  TOV          15914 non-null  int64 
 14  STL          15914 non-null  int64 
 15  BLK          15914 non-null  int64 
 16  PTS          15914 non-null  int64 
 17  PLUS_MINUS   15914 non-null  int64 
 18  PLAYER_NAME  15914 non-null  object
 19  PLAYER_ID    15914 non-nu

## Convert game date and sort records

Sorting by player and date makes the dataset easier to inspect and prepares it for later transformations.

In [6]:
df["GAME_DATE"] = pd.to_datetime(df["GAME_DATE"])
df = df.sort_values(["PLAYER_NAME", "GAME_DATE"]).reset_index(drop=True)

df[["PLAYER_NAME", "GAME_DATE", "MATCHUP", "PTS", "MIN", "FGA"]].head(10)

,PLAYER_NAME,GAME_DATE,MATCHUP,PTS,MIN,FGA
0,A.J. Lawson,2023-12-01,DAL vs. MEM,1,4,1
1,A.J. Lawson,2023-12-02,DAL vs. OKC,12,19,10
2,A.J. Lawson,2023-12-06,DAL vs. UTA,4,7,2
3,A.J. Lawson,2023-12-08,DAL @ POR,2,7,3
4,A.J. Lawson,2023-12-11,DAL @ MEM,4,14,7
5,A.J. Lawson,2023-12-12,DAL vs. LAL,0,0,0
6,A.J. Lawson,2023-12-14,DAL vs. MIN,2,2,1
7,A.J. Lawson,2023-12-16,DAL @ POR,0,1,0
8,A.J. Lawson,2023-12-18,DAL @ DEN,6,15,4
9,A.J. Lawson,2023-12-20,DAL vs. LAC,0,1,0


## Date coverage

Check the time span covered by the raw data.

In [7]:
print("Min date:", df["GAME_DATE"].min().date())
print("Max date:", df["GAME_DATE"].max().date())

Min date: 2023-10-24
Max date: 2024-04-14


## Example players

Preview a few player names included in the dataset.

In [8]:
sorted(df["PLAYER_NAME"].unique())[:20]

['A.J. Lawson',
 'AJ Green',
 'Aaron Gordon',
 'Aaron Holiday',
 'Aaron Nesmith',
 'Al Horford',
 'Alex Caruso',
 'Amir Coffey',
 'Andre Drummond',
 'Andre Jackson Jr.',
 'Andrew Nembhard',
 'Anthony Black',
 'Anthony Davis',
 'Anthony Edwards',
 'Anthony Gill',
 'Austin Reaves',
 'Ayo Dosunmu',
 'Bam Adebayo',
 'Bennedict Mathurin',
 'Bilal Coulibaly']

## Number of games per player

This helps identify players with very short histories and players with enough data for rolling features.

In [9]:
df["PLAYER_NAME"].value_counts().head(20)

PLAYER_NAME
Buddy Hield                 84
Paul Reed                   82
Payton Pritchard            82
Christian Braun             82
Chet Holmgren               82
Bobby Portis                82
Nickeil Alexander-Walker    82
Mikal Bridges               82
Harrison Barnes             82
Jalen Green                 82
Austin Reaves               82
Georges Niang               82
Josh Hart                   81
Michael Porter Jr.          81
Naz Reid                    81
Donte DiVincenzo            81
Cole Anthony                81
Corey Kispert               80
Paolo Banchero              80
Josh Giddey                 80
Name: count, dtype: int64

## Missing values check

This step helps identify columns that may require special handling later.

In [10]:
df.isna().sum().sort_values(ascending=False)

Game_ID        0
GAME_DATE      0
MATCHUP        0
WL             0
MIN            0
FGM            0
FGA            0
FG3M           0
FG3A           0
FTM            0
FTA            0
REB            0
AST            0
TOV            0
STL            0
BLK            0
PTS            0
PLUS_MINUS     0
PLAYER_NAME    0
PLAYER_ID      0
dtype: int64

## Basic descriptive statistics

A quick summary of selected numeric variables in the raw dataset.

In [11]:
numeric_cols = ["MIN", "FGA", "FG3A", "FTA", "REB", "AST", "TOV", "PTS"]
df[numeric_cols].describe()

,MIN,FGA,FG3A,FTA,REB,AST,TOV,PTS
count,15914.000000,15914.000000,15914.000000,15914.000000,15914.000000,15914.000000,15914.000000,15914.000000
mean,24.982091,9.346990,3.656717,2.334234,4.522999,2.803695,1.343597,12.183549
std,10.321422,6.294646,3.168567,2.958831,3.546445,2.762820,1.414907,9.240079
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,18.000000,4.000000,1.000000,0.000000,2.000000,1.000000,0.000000,5.000000
50%,26.000000,8.000000,3.000000,2.000000,4.000000,2.000000,1.000000,11.000000
75%,33.000000,13.000000,6.000000,4.000000,6.000000,4.000000,2.000000,17.000000
max,54.000000,47.000000,23.000000,32.000000,31.000000,23.000000,9.000000,73.000000


## Raw data preview for one sample player

This makes it easier to understand how player game logs look before feature engineering.

In [12]:
sample_player = df["PLAYER_NAME"].value_counts().index[0]
df[df["PLAYER_NAME"] == sample_player][["GAME_DATE", "MATCHUP", "PTS", "MIN", "FGA"]].head(10)

,GAME_DATE,MATCHUP,PTS,MIN,FGA
2070,2023-10-25,IND vs. WAS,14,25,9
2071,2023-10-28,IND @ CLE,10,19,12
2072,2023-10-30,IND vs. CHI,11,19,12
2073,2023-11-01,IND @ BOS,7,24,8
2074,2023-11-03,IND vs. CLE,14,30,9
2075,2023-11-04,IND vs. CHA,19,26,15
2076,2023-11-06,IND vs. SAS,19,21,8
2077,2023-11-08,IND vs. UTA,10,22,16
2078,2023-11-09,IND vs. MIL,11,22,14
2079,2023-11-12,IND @ PHI,16,21,14


## Observations

At this stage:
- the project uses a full-league raw dataset
- data is available at the player-game level
- the next step is to transform these raw logs into rolling and trend-based modeling features